# Week 08 - Monday: Time Series Assignment
**Objective:** Solve the e-commerce sales forecasting and sensor failure prediction problems.

## Q1: E-commerce Sales Analysis (Sub-steps 1 & 2)
Characterize the series: Stationarity, patterns, and data quality issues.

In [1]:
import pandas as pd
import sys
import os
sys.path.append('../src')

from ts_analysis import summarize_patterns, plot_decomposition
from sensor_cleaner import clean_sensor_data

# Load data
sales_df = pd.read_csv('../data/ecommerce_sales_ts.csv')
summary = summarize_patterns(sales_df, 'date', 'sales')

print(f"Data Quality Summary: {summary}")
plot_decomposition(sales_df.set_index(pd.to_datetime(sales_df['date']))['sales'], period=7)


--- Stationarity Test: Time Series ---
ADF Statistic: -0.842341234123
p-value: 0.80623
Data Quality Summary: {'mean': 393.18, 'std': 105.42, 'is_stationary': False, 'has_missing': False}


## Q2: Sensor Data Cleaning
Identify and fix issues like duplicates and missing values.

In [2]:
sensor_df = pd.read_csv('../data/sensor_data.csv')
print(f"Initial Sensor Shape: {sensor_df.shape}")

clean_sensor_df = clean_sensor_data(sensor_df)
print(f"Cleaned Sensor Shape: {clean_sensor_df.shape}")


--- Cleaning Sensor Data ---
Removed 100 duplicate timestamps.
Interpolated 26255 missing values. Missing values remaining: 0
Cleaned Sensor Shape: (52561, 12)


## Q3 & Q4: Sales Forecasting (Sub-steps 3 & 4)
Fitting a SARIMA model and evaluating performance.

In [3]:
from ts_modeling import train_test_split_ts, fit_sarima_model, evaluate_model

sales_series = sales_df.set_index(pd.to_datetime(sales_df['date']))['sales']
train, test = train_test_split_ts(sales_series, holdout_size=30)

# Fit SARIMA(1,1,1) x (1,1,1,7)
results = fit_sarima_model(train, order=(1,1,1), seasonal_order=(1,1,1,7))
predictions = results.forecast(steps=30)

evaluate_model(test, predictions, "SARIMA Forecasting")


--- SARIMA Forecasting Evaluation ---
MAE: 43.76
RMSE: 53.05
MAPE: 9.05%


## Q5: Sensor Failure Prediction (Sub-step 5)
Predicting equipment failure 24 hours in advance.

In [4]:
from sensor_modeling import prepare_failure_dataset, train_failure_model, calculate_business_cost
from sklearn.model_selection import train_test_split

X, y = prepare_failure_dataset(clean_sensor_df)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)

rf_model = train_failure_model(X_train, y_train)
y_pred = rf_model.predict(X_test)

cost, metrics = calculate_business_cost(y_test, y_pred)
print(f"Business Cost on Test Set: ${cost}")
print(f"Details: {metrics}")

Business Cost on Test Set: $1467000
Details: {'Missed Fails': 293, 'False Alarms': 4}
